# NESTFUL eval — Colab + Google Drive

1. Nahraj na Drive: `Tool-R0/run.py` + tento notebook
2. **Runtime → Restart session → A100 GPU**
3. **Run all**

**Colab má CUDA 12.x** — notebook instaluje **cu129** wheel (ne `pip install vllm`).

**Logy během evaluace:** buňka eval streamuje `[loop]` a `[progress]` řádky. Průběh je i na Drive v `*_multiturn_progress.json` (poslední buňku můžeš spustit znovu i za běhu).

In [ ]:
# ============ KONFIGURACE ============
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Tool-R0"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# Celý benchmark: MAX_TASKS = None (1861 úloh), NUM_ROLLOUTS = 8 → ~14888 rolloutů
MAX_TASKS = None          # pilot: 50
NUM_ROLLOUTS = 8          # pilot: 2
MAX_STEPS = 10
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 2048
MAX_MODEL_LEN = 12288
GPU_MEMORY_UTILIZATION = 0.90
IBM_CALL_TIMEOUT = 30
ADVANCE_LOG_EVERY = 100   # log každých N rolloutů během tool dispatch (500=méně řádků)
SEED = 0
HF_TOKEN = None

In [ ]:
import os, subprocess, sys, shutil

def sh(cmd, check=True):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=check)

from google.colab import drive
drive.mount("/content/drive")

WORK_DIR = DRIVE_PROJECT_DIR.rstrip("/")
RUN_PY_DRIVE = os.path.join(WORK_DIR, "run.py")
OUTPUT_DIR = os.path.join(WORK_DIR, "nestful_results")
NESTFUL_REPO_DIR = "/content/nestful_repo"
LOCAL_RUN_DIR = "/content/nestful_eval"
LOCAL_RUN_PY = os.path.join(LOCAL_RUN_DIR, "run.py")
os.makedirs(LOCAL_RUN_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isfile(RUN_PY_DRIVE):
    raise FileNotFoundError(f"Na Drive chybí: {RUN_PY_DRIVE}")
shutil.copy2(RUN_PY_DRIVE, LOCAL_RUN_PY)
print(f"Drive:   {RUN_PY_DRIVE}")
print(f"Lokálně: {LOCAL_RUN_PY}")
print(f"Výstup:  {OUTPUT_DIR}")

In [ ]:
sh("nvidia-smi || true", check=False)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Zapni GPU: Runtime → Change runtime type → A100")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch CUDA: {torch.version.cuda}")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {vram_gb:.1f} GB")
if vram_gb >= 35:
    MAX_MODEL_LEN = 12288
    GPU_MEMORY_UTILIZATION = 0.90
elif vram_gb >= 20:
    MAX_MODEL_LEN = 8192
    GPU_MEMORY_UTILIZATION = 0.85
else:
    MAX_MODEL_LEN = 4096
    GPU_MEMORY_UTILIZATION = 0.70
print(f"→ MAX_MODEL_LEN={MAX_MODEL_LEN}")

In [ ]:
# ============ INSTALACE + SMOKE TEST ============
# PyPI vLLM 0.22 = CUDA 13 (cu130). Colab A100 = CUDA 12.x → potřebuje cu129 wheel.
import gc, platform, subprocess, sys, textwrap, importlib
import torch
from packaging import version

PY = sys.executable
VLLM_VER = "0.22.0"
TORCH_CUDA = torch.version.cuda or ""
ARCH = "aarch64" if platform.machine() == "aarch64" else "x86_64"

def pip(*args, check=False):
    cmd = [PY, "-m", "pip", *args]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

def vllm_wheel_url(cu_tag: str) -> str:
    return (
        f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VER}/"
        f"vllm-{VLLM_VER}+cu{cu_tag}-cp38-abi3-manylinux_2_28_{ARCH}.whl"
    )

def install_vllm_for_colab():
    pip("install", "-q", "-U", "pip", "packaging", "datasets>=3.5.0", "huggingface_hub>=0.30.0")
    pip("install", "-q", "-U", "transformers")

    if TORCH_CUDA.startswith("13"):
        print(f"PyTorch CUDA {TORCH_CUDA} → pip vllm (cu130)")
        pip("install", "-q", "-U", "vllm")
        return

    # Colab 2026: CUDA 12.8/12.9 — cu129 wheel (cu130 z PyPI nefunguje)
    cu_tag = "129"
    wheel = vllm_wheel_url(cu_tag)
    pt_index = f"https://download.pytorch.org/whl/cu{cu_tag}"
    print(f"PyTorch CUDA {TORCH_CUDA} → vLLM {VLLM_VER}+cu{cu_tag}")
    print(f"  {wheel}")
    pip("uninstall", "-y", "vllm", check=False)
    rc = pip("install", "-q", wheel, "--extra-index-url", pt_index).returncode
    if rc != 0:
        print("GitHub wheel selhal — fallback pip + cu129 index")
        pip("install", "-q", "vllm", "--extra-index-url", pt_index)

print("=" * 60)
print("Instalace vLLM pro CUDA verzi Colabu")
print("=" * 60)
install_vllm_for_colab()

import transformers
importlib.invalidate_caches()
import vllm

print(f"vllm {vllm.__version__}")
print(f"transformers {transformers.__version__}")

if version.parse(transformers.__version__) < version.parse("4.51.0"):
    raise RuntimeError(f"transformers {transformers.__version__} moc staré")
if version.parse(vllm.__version__) < version.parse("0.8.5"):
    raise RuntimeError(f"vLLM {vllm.__version__} moc staré")

try:
    from vllm import LLM
except (ModuleNotFoundError, ImportError) as exc:
    if "libcudart" not in str(exc):
        raise
    print(f"Špatný CUDA wheel ({exc}) — reinstall cu129 ...")
    install_vllm_for_colab()
    importlib.invalidate_caches()
    from vllm import LLM

from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL, trust_remote_code=True)
print(f"AutoConfig model_type={cfg.model_type} OK")

print(f"\nNačítám {MODEL} (max_model_len={MAX_MODEL_LEN}) ...")
_test = textwrap.dedent(f"""
import gc, torch
from vllm import LLM
llm = LLM(model={MODEL!r}, tensor_parallel_size=1,
          gpu_memory_utilization={GPU_MEMORY_UTILIZATION},
          max_model_len={MAX_MODEL_LEN}, enforce_eager=True,
          trust_remote_code=True)
del llm
gc.collect()
torch.cuda.empty_cache()
print("Qwen3 smoke test OK")
""")
r = subprocess.run([PY, "-c", _test], capture_output=True, text=True)
if r.stdout:
    print(r.stdout.strip())
if r.returncode != 0:
    if r.stderr:
        print("--- stderr ---")
        print(r.stderr[-8000:])
    raise RuntimeError("vLLM smoke test selhal")

VLLM_SMOKE_OK = True
print("\n✓ Hotovo — spusť evaluaci")

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except Exception:
        login()

In [ ]:
SENTINEL = "/content/nestful_repo/data_v2/executable_functions/func_file_map.json"
if not os.path.isfile(SENTINEL):
    sh("rm -rf /content/nestful_repo")
    sh("git clone --depth 1 https://github.com/IBM/NESTFUL.git /content/nestful_repo")
else:
    print("IBM repo OK")

from datasets import load_dataset
load_dataset("ibm-research/nestful", split="train[:1]")
print("HF dataset OK")

In [ ]:
# ============ SPUŠTĚNÍ EVALUACE (stream logů v reálném čase) ============
if not globals().get("VLLM_SMOKE_OK"):
    raise RuntimeError("Nejdřív spusť buňku Instalace + smoke test!")

cmd = [
    sys.executable, "-u", LOCAL_RUN_PY,  # -u = unbuffered stdout
    "--model", MODEL,
    "--output-dir", OUTPUT_DIR,
    "--num-rollouts", str(NUM_ROLLOUTS),
    "--max-steps", str(MAX_STEPS),
    "--temperature", str(TEMPERATURE),
    "--top-p", str(TOP_P),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--ibm-call-timeout", str(IBM_CALL_TIMEOUT),
    "--advance-log-every", str(ADVANCE_LOG_EVERY),
    "--seed", str(SEED),
    "--nestful-repo-dir", NESTFUL_REPO_DIR,
]
if MAX_TASKS is not None:
    cmd.extend(["--max-tasks", str(MAX_TASKS)])

n_tasks = MAX_TASKS if MAX_TASKS is not None else 1861
profile = MODEL.replace("/", "__").lower()
PROGRESS_JSON = os.path.join(
    OUTPUT_DIR, f"{profile}_multiturn_progress.json"
)

print(f"Eval: {n_tasks} úloh × {NUM_ROLLOUTS} rolloutů = {n_tasks * NUM_ROLLOUTS} rolloutů")
print(f"Logy: [loop] = krok evaluace, [progress] = souhrn + ETA")
print(f"Průběh na Drive: {PROGRESS_JSON}")
print(" ".join(cmd), "\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(
    cmd,
    cwd=LOCAL_RUN_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
returncode = proc.wait()
if returncode != 0:
    raise RuntimeError(f"run.py exit {returncode} — scrolluj nahoru pro Traceback")
print("\n✓ Evaluace dokončena")

In [ ]:
import json, glob, os

# Průběh (aktualizuje se po každém kroku během běžící evaluace)
progress_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_multiturn_progress.json")))
if progress_files:
    with open(progress_files[-1]) as f:
        p = json.load(f)
    print("=== Průběh (progress.json) ===")
    for k in [
        "step", "max_steps", "rollouts_done", "rollouts_total",
        "passed", "failed", "accuracy_percent",
        "elapsed_seconds", "eta_seconds", "stop_reason_breakdown",
    ]:
        if k in p:
            print(f"  {k}: {p[k]}")
    print(f"\nSoubor: {progress_files[-1]}")

summaries = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_multiturn_summary.json")))
if summaries:
    with open(summaries[-1]) as f:
        s = json.load(f)
    print("\n=== Finální summary ===")
    for k in [
        "model", "total_rollouts", "passed", "failed",
        "final_answer_accuracy", "elapsed_seconds", "stop_reason_breakdown",
    ]:
        if k in s:
            print(f"  {k}: {s[k]}")
    print(f"\nSoubor: {summaries[-1]}")
elif not progress_files:
    print("Summary ani progress nenalezen")